# rational vs. other sample rate converters — measured

The async comparison asks what clock recovery costs; the bridge comparison
asks the synchronous question at one ratio pair. This one asks it at four
ratios of `rational`'s vocabulary — **↑2 (48 → 96 kHz), ↓2 (96 → 48),
3/2 (32 → 48) and 2/3 (48 → 32)** — the by-2 pair every family has and the
mixed pair that a single Nyquist stage serves. The same AES17-style
instrument is applied identically to every subject, through the **shipping
C++** (the C ABI via `tap_sr_rational_py.py`) and the real libraries
(libsamplerate through its Python binding; soxr, r8brain and SpeexDSP through
the family's C shims in `tools/compare/shim`), never a Python
re-implementation.

The subjects, converting the same signals at the same ratio:

| Subject | What it is | Settings measured |
|---|---|---|
| `rational` | this engine, one L-th-band (Nyquist) stage per ratio | `economy` (70 dB, passband 3/8 of the lower rate), `balanced`, `transparent` (120 dB, 5/12); `economy` and `transparent` in Q15 |
| libsamplerate | resampler library, five fixed converters | `SINC_FASTEST`, `SINC_MEDIUM`, `SINC_BEST` |
| soxr | resampler library, FFT-based, custom quality spec | 16-bit at a 0.75 passband, 20-bit at 0.80 (custom), `VHQ` |
| r8brain-free-src | resampler library, FFT block convolution, continuous knobs | 70 dB at a 16 % band, 120 dB at a 10 % band, `CDSPResampler24` |
| SpeexDSP | resampler library, polyphase, the one with a fixed-point build | quality 2, 9, 10 (float); 3, 10 (fixed-point) |

**The matching rule** (stated once, applied by the sweep below): a
competitor's *matched* setting for a tier is its cheapest whose measured
stopband is at least the tier's (70 dB for `economy`, 120 dB for
`transparent`) and whose passband droops no more than 0.1 dB at the tier's
edge (3/8 and 5/12 of the lower rate), at **every one of the four ratios**;
its *frontier* is its best. A Nyquist design's transition band is symmetric
about the lower Nyquist, so a decimator's stopband begins at (lower rate −
passband edge): tones in the transition band alias above the passband edge
by contract, and the sweep's stopband probes sit above that.

In [1]:
import ctypes, pathlib, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt
import figure_digest  # prints a data digest per figure (A/B gates)
figure_digest.install()
import samplerate
from tap_sr_rational_py import Chain, version

# (input rate, output rate, the chain constant's name); the by-2 pair at 48 <-> 96
# and the mixed pair at 32 <-> 48.
RATIOS = {"↑2 48 → 96": (48000.0, 96000.0, "up_2"), "↓2 96 → 48": (96000.0, 48000.0, "down_2"),
          "3/2 32 → 48": (32000.0, 48000.0, "ratio_3_2"), "2/3 48 → 32": (48000.0, 32000.0, "ratio_2_3")}
CHAIN_OF = {(fi, fo): ch for fi, fo, ch in RATIOS.values()}
DIRECTIONS = {k: (v[0], v[1]) for k, v in RATIOS.items()}   # name -> (input rate, output rate)
def edges(src, dst):
    # The tiers' passband edges at this ratio: fractions of the lower rate.
    r_min = min(src, dst)
    return {"economy": 3.0 / 8.0 * r_min, "transparent": 5.0 / 12.0 * r_min}

REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
TOOLS_DIR = REPO / "build"

def _find_dso(stem):
    for name in (f"lib{stem}.so", f"lib{stem}.dylib", f"{stem}.dll"):
        hits = sorted(TOOLS_DIR.rglob(name))
        if hits:
            return hits[0]
    return None

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout); print(r.stderr, file=sys.stderr)
        raise RuntimeError("command failed: " + " ".join(cmd))

# The family's C shims over the pinned r8brain-free-src and SpeexDSP sources
# and the system soxr (tools/compare/shim): r8brain and SpeexDSP have no
# maintained Python binding, and python-soxr exposes only the named recipes,
# not the custom quality spec the matched row needs. Always (re)build: an
# incremental build is quick, and a library left over from an older checkout
# would silently measure old code.
_run(["cmake", "-B", str(TOOLS_DIR), "-S", str(REPO), "-DCMAKE_BUILD_TYPE=Release",
      "-DTAP_SR_BUILD_COMPARE_SHIM=ON", "-DTAP_SR_BUILD_TESTS=OFF", "-DTAP_SR_BUILD_EXAMPLES=OFF"])
_run(["cmake", "--build", str(TOOLS_DIR), "--target", "tap_sr_cmp_r8b_shim", "tap_sr_cmp_speex_shim",
      "tap_sr_cmp_soxr_shim", "--config", "Release", "-j"])
R8B_DSO, SPEEX_DSO, SOXR_DSO = (_find_dso(s) for s in ("tap_sr_cmp_r8b_shim", "tap_sr_cmp_speex_shim", "tap_sr_cmp_soxr_shim"))
assert R8B_DSO and SPEEX_DSO and SOXR_DSO
_FP = ctypes.POINTER(ctypes.c_float)

_r8b = ctypes.CDLL(str(R8B_DSO))
_r8b.tap_sr_cmp_r8b_oneshot.restype = ctypes.c_int
_r8b.tap_sr_cmp_r8b_oneshot.argtypes = [_FP, ctypes.c_int, ctypes.c_double, ctypes.c_double, ctypes.c_double,
                                        ctypes.c_double, ctypes.c_int, _FP, ctypes.c_int]
_r8b.tap_sr_cmp_r8b_latency_frames.restype = ctypes.c_int
_r8b.tap_sr_cmp_r8b_latency_frames.argtypes = [ctypes.c_double] * 4 + [ctypes.c_int]
_speex = ctypes.CDLL(str(SPEEX_DSO))
_speex.tap_sr_cmp_speex_oneshot.restype = ctypes.c_int
_speex.tap_sr_cmp_speex_oneshot.argtypes = [_FP, ctypes.c_int, ctypes.c_double, ctypes.c_double, ctypes.c_int,
                                            ctypes.c_int, _FP, ctypes.c_int]
_speex.tap_sr_cmp_speex_latency_frames.restype = ctypes.c_int
_speex.tap_sr_cmp_speex_latency_frames.argtypes = [ctypes.c_double, ctypes.c_double, ctypes.c_int, ctypes.c_int]
_soxr = ctypes.CDLL(str(SOXR_DSO))
_soxr.tap_sr_cmp_soxr_oneshot.restype = ctypes.c_int
_soxr.tap_sr_cmp_soxr_oneshot.argtypes = [_FP, ctypes.c_int, ctypes.c_double, ctypes.c_double, ctypes.c_int,
                                          ctypes.c_double, ctypes.c_double, _FP, ctypes.c_int]
_soxr.tap_sr_cmp_soxr_delay_frames.restype = ctypes.c_double
_soxr.tap_sr_cmp_soxr_delay_frames.argtypes = [ctypes.c_double, ctypes.c_double, ctypes.c_int, ctypes.c_double, ctypes.c_double]
SOXR_HQ, SOXR_VHQ = 4, 6   # soxr.h: SOXR_20_BITQ, SOXR_28_BITQ

def _nout(n, src, dst):
    return int(round(n * dst / src))

# --- one whole-buffer conversion per subject: mono float32 in, float32 out ---
def via_rational(x, src, dst, profile="economy", fmt="float"):
    conv = Chain(CHAIN_OF[(src, dst)], profile=profile, fmt=fmt)
    if fmt == "q15":   # a Q15 caller's interface: round and saturate in, scale out
        q = np.clip(np.round(np.asarray(x, np.float64) * 32767.0), -32768, 32767).astype(np.int16)
        return np.concatenate([conv.process(q), conv.flush()]).astype(np.float32) / 32768.0
    return np.concatenate([conv.process(np.asarray(x, np.float32)), conv.flush()])

def via_lsr(x, src, dst, converter):
    return samplerate.resample(np.asarray(x, np.float32), dst / src, converter).astype(np.float32)

def via_soxr(x, src, dst, recipe=-1, precision=0.0, passband_end=0.0):
    x = np.ascontiguousarray(x, np.float32)
    y = np.zeros(_nout(len(x), src, dst) + 16, np.float32)
    n = _soxr.tap_sr_cmp_soxr_oneshot(x.ctypes.data_as(_FP), len(x), src, dst, recipe, precision, passband_end,
                                      y.ctypes.data_as(_FP), len(y))
    if n <= 0:
        raise ValueError(f"soxr rejected recipe={recipe} precision={precision} passband_end={passband_end}")
    return y[:n]

def via_r8brain(x, src, dst, trans_band_pct, atten_db):
    x = np.ascontiguousarray(x, np.float32)
    y = np.zeros(_nout(len(x), src, dst), np.float32)
    assert _r8b.tap_sr_cmp_r8b_oneshot(x.ctypes.data_as(_FP), len(x), src, dst, trans_band_pct, atten_db, 0,
                                       y.ctypes.data_as(_FP), len(y)) == 0
    return y

def via_speex(x, src, dst, quality, fixed_point=False):
    x = np.ascontiguousarray(x, np.float32)
    y = np.zeros(_nout(len(x), src, dst), np.float32)
    assert _speex.tap_sr_cmp_speex_oneshot(x.ctypes.data_as(_FP), len(x), src, dst, quality, int(fixed_point),
                                           y.ctypes.data_as(_FP), len(y)) == 0
    return y

# --- streaming latency, in input frames before the first output frame ---
def lsr_latency_frames(src, dst, converter):
    rs = samplerate.Resampler(converter, channels=1)
    fed = 0
    while fed < 100000:
        out = rs.process(np.zeros(1, np.float32), dst / src, end_of_input=False)
        fed += 1
        if len(out) > 0:
            return fed
    raise RuntimeError("no output")

def rational_latency_output_frames(src, dst, profile="economy"):
    return float(Chain(CHAIN_OF[(src, dst)], profile=profile).latency_output_frames)

print(f"rational C ABI version {version()}; shims: {R8B_DSO.name}, {SPEEX_DSO.name}, {SOXR_DSO.name}; "
      f"libsamplerate via samplerate {samplerate.__version__}")

rational C ABI version (0, 6, 0); shims: libtap_sr_cmp_r8b_shim.so, libtap_sr_cmp_speex_shim.so, libtap_sr_cmp_soxr_shim.so; libsamplerate via samplerate 0.2.4


## The instrument, and its calibration

The same AES17-style measurement as the async and bridge comparisons: THD+N is a
997 Hz tone at −1 dBFS with the fundamental removed by an exact
least-squares fit at the phase-slope-refined frequency plus a ±20 Hz notch,
the residual integrated 20 Hz–20 kHz; dynamic range is the same at −60 dBFS,
A-weighted. The **24-bit interface** columns quantize every subject's output
to 24 bits, the condition under which software numbers are comparable to
silicon datasheets. The residual is integrated 20 Hz–20 kHz, or to 1 kHz
below the output Nyquist where that is lower (32 kHz out: 20 Hz–15 kHz).
The instrument is validated against synthetic signals with known answers at
every output rate before it measures anything.

In [2]:
def a_weight(f):
    f2 = np.square(f)
    num = (12194.0**2) * f2 * f2
    den = ((f2 + 20.6**2) * np.sqrt((f2 + 107.7**2) * (f2 + 737.9**2)) * (f2 + 12194.0**2))
    ra = np.divide(num, den, out=np.zeros_like(f2), where=den > 0)
    ref = (12194.0**2 * 1000.0**4) / ((1000.0**2 + 20.6**2)
          * np.sqrt((1000.0**2 + 107.7**2) * (1000.0**2 + 737.9**2)) * (1000.0**2 + 12194.0**2))
    return ra / ref

def refine_freq(x, f0, fs, win=8192, iters=3):
    n = (len(x) // win) * win
    tt = np.arange(n)
    for _ in range(iters):
        ph = []
        for k in range(n // win):
            sl = slice(k * win, (k + 1) * win)
            A = np.column_stack([np.sin(2 * np.pi * f0 / fs * tt[sl]), np.cos(2 * np.pi * f0 / fs * tt[sl])])
            c, *_ = np.linalg.lstsq(A, x[sl].astype(np.float64), rcond=None)
            ph.append(np.arctan2(c[1], c[0]))
        ph = np.unwrap(ph)
        tk = (np.arange(len(ph)) + 0.5) * win / fs
        f0 += np.polyfit(tk, ph, 1)[0] / (2 * np.pi)
    return f0

def thdn_db(x, f0_nominal, fs, weighted=False, notch_hz=20.0, band=(20.0, 20000.0)):
    # AES17-style THD+N in dB relative to the fundamental (negative).
    x = x.astype(np.float64)
    f0 = refine_freq(x, f0_nominal, fs)
    tt = np.arange(len(x))
    A = np.column_stack([np.sin(2 * np.pi * f0 / fs * tt), np.cos(2 * np.pi * f0 / fs * tt)])
    c, *_ = np.linalg.lstsq(A, x, rcond=None)
    fund_rms = np.hypot(c[0], c[1]) / np.sqrt(2)
    resid = x - A @ c
    R = np.fft.rfft(resid)
    f = np.fft.rfftfreq(len(resid), 1 / fs)
    mask = (f >= band[0]) & (f <= band[1]) & (np.abs(f - f0) > notch_hz)
    w = a_weight(f) if weighted else np.ones_like(f)
    resid_rms = np.sqrt(np.sum(np.abs(R[mask] * w[mask])**2) * 2 / len(resid)**2)
    return 20 * np.log10(resid_rms / fund_rms)

def fit_amp(y, f, fs):
    # Amplitude of the component at a known frequency (least squares).
    y = np.asarray(y, np.float64)
    t = np.arange(len(y))
    A = np.column_stack([np.sin(2 * np.pi * f / fs * t), np.cos(2 * np.pi * f / fs * t)])
    c, *_ = np.linalg.lstsq(A, y, rcond=None)
    return np.hypot(c[0], c[1])

def mid(y, fs, seconds):
    # The middle of the stream: one-shot converters flush a filter tail at
    # the end, and the start carries the delay they remove.
    n = int(seconds * fs)
    s = (len(y) - n) // 2
    return np.asarray(y[s:s + n], np.float32)

def q24(y):
    # Round to a 24-bit interface, undithered -- what silicon presents at its pins.
    return np.round(np.asarray(y, np.float64) * 8388608.0) / 8388608.0

def tone(f, seconds, fs, amp_dbfs):
    return (10 ** (amp_dbfs / 20) * np.sin(2 * np.pi * f / fs * np.arange(int(seconds * fs)))).astype(np.float32)

rng = np.random.default_rng(11)
for fs in (32000.0, 48000.0, 96000.0):
    n = int(8 * fs)
    sine = tone(997.003, 8.0, fs, -1.0)
    for noise_db in (-100.0, -130.0):
        sig = (sine + 10 ** (noise_db / 20) * rng.standard_normal(n)).astype(np.float32)
        band_hi = min(20000.0, fs / 2 - 1000.0)   # the output Nyquist bounds the band at 32 kHz
        got = thdn_db(sig, 997.0, fs, band=(20.0, band_hi))
        inband = (band_hi - 20.0) / (fs / 2)
        expect = noise_db - 20 * np.log10(10 ** (-1 / 20) / np.sqrt(2)) + 10 * np.log10(inband)
        print(f"fs {fs:.0f}: injected {noise_db:6.1f} dBFS noise -> measured THD+N {got:7.2f} dB (expect {expect:7.2f})")
        assert abs(got - expect) < 0.5
print("instrument calibrated at every output rate")

fs 32000: injected -100.0 dBFS noise -> measured THD+N  -96.30 dB (expect  -96.28)
fs 32000: injected -130.0 dBFS noise -> measured THD+N -126.29 dB (expect -126.28)


fs 48000: injected -100.0 dBFS noise -> measured THD+N  -96.79 dB (expect  -96.79)


fs 48000: injected -130.0 dBFS noise -> measured THD+N -126.76 dB (expect -126.79)


fs 96000: injected -100.0 dBFS noise -> measured THD+N  -99.80 dB (expect  -99.80)
fs 96000: injected -130.0 dBFS noise -> measured THD+N -129.79 dB (expect -129.80)
instrument calibrated at every output rate


## The matched settings, measured

Each library's quality knob is swept at all four ratios. Two numbers per
setting decide the match: the **passband droop** at the tier's edge (the
fitted gain of a tone at the edge, relative to 997 Hz) and the **worst
spurious product**, found at the exact frequencies the arithmetic puts them —
every |a·f_in + b·f_out ± f| below the output Nyquist for small a, b — from
tones in the passband (images) and, for the decimations, from tones in the
stopband above (lower rate − passband edge), whose aliases land inside the
passband. The rule then picks, per tier, the cheapest setting that holds
both at every ratio. The picks are asserted against the constants the host
and embedded benches compile in, so the benches cannot drift from this
measurement.

In [3]:
def droop_db(conv, src, dst, f):
    y = mid(conv(tone(f, 1.5, src, -6.0), src, dst), dst, 1.0)
    ref = mid(conv(tone(997.0, 1.5, src, -6.0), src, dst), dst, 1.0)
    return 20 * np.log10(fit_amp(y, f, dst) / fit_amp(ref, 997.0, dst))

def spur_candidates(f, src, dst):
    # Every |a*f_in + b*f_out +- f| below the output Nyquist, for small a and b,
    # away from the tone itself.
    c = set()
    for a in range(-4, 5):
        for b in range(-4, 5):
            for sgn in (f, -f):
                v = abs(a * src + b * dst + sgn)
                if 50.0 < v < dst / 2 - 50.0 and abs(v - f) > 200.0:
                    c.add(round(v, 3))
    return sorted(c)

def worst_product_db(conv, src, dst):
    edge = edges(src, dst)["economy"]
    probes = [0.6 * edge, 0.8 * edge, edge]           # passband tones: images
    if src > dst:                                      # decimations: stopband tones, aliasing into the passband
        lo, hi = dst - edge, src / 2
        probes += [lo + 0.05 * (hi - lo), lo + 0.5 * (hi - lo), hi - 0.05 * (hi - lo)]
    worst = -400.0
    for f in probes:
        y = mid(conv(tone(f, 1.5, src, -6.0), src, dst), dst, 1.0)
        for fp in spur_candidates(f, src, dst):
            worst = max(worst, 20 * np.log10(fit_amp(y, fp, dst) / 10 ** (-6 / 20) + 1e-30))
    return worst

# Candidate settings per library, cheapest first (the order the rule picks in).
CANDIDATES = {
    "libsamplerate": [(c, lambda x, s, d, c=c: via_lsr(x, s, d, c)) for c in ("sinc_fastest", "sinc_medium", "sinc_best")],
    "soxr": [(f"{p:.0f}-bit @ {pe:.3f}", lambda x, s, d, p=p, pe=pe: via_soxr(x, s, d, -1, p, pe))
             for p in (16.0, 20.0) for pe in (0.750, 0.800, 0.833, 0.870)]
            + [("HQ", lambda x, s, d: via_soxr(x, s, d, SOXR_HQ)), ("VHQ", lambda x, s, d: via_soxr(x, s, d, SOXR_VHQ))],
    "r8brain": [(f"{a:.0f} dB @ {tb:g} %", lambda x, s, d, a=a, tb=tb: via_r8brain(x, s, d, tb, a))
                for a, tbs in ((70.0, (25.0, 20.0, 18.0, 16.0, 14.0, 12.0)), (120.0, (16.0, 14.0, 12.0, 10.0, 8.0, 6.0)))
                for tb in tbs],
    "SpeexDSP float": [(f"q{q}", lambda x, s, d, q=q: via_speex(x, s, d, q)) for q in range(1, 11)],
    "SpeexDSP fixed": [(f"q{q}", lambda x, s, d, q=q: via_speex(x, s, d, q, True)) for q in (1, 2, 3, 4, 9, 10)],
}
TIERS = {"economy": 70.0, "transparent": 120.0}   # stopband dB; the passband edge is edges()'s, per ratio
DROOP_MAX_DB = 0.1

sweep = {}   # (library, setting) -> per ratio: (droop at the economy edge, at the transparent edge, worst)
for lib, cands in CANDIDATES.items():
    print(f"--- {lib}")
    print(f"{'setting':18s} " + "   ".join(f"{d:>11s}: eco    tr  worst" for d in DIRECTIONS))
    for name, conv in cands:
        row = {}
        for dname, (src, dst) in DIRECTIONS.items():
            e = edges(src, dst)
            try:
                row[dname] = (droop_db(conv, src, dst, e["economy"]), droop_db(conv, src, dst, e["transparent"]),
                              worst_product_db(conv, src, dst))
            except ValueError:
                row[dname] = None
        sweep[(lib, name)] = row
        print(f"{name:18s} " + "   ".join(("     rejected by soxr     " if r is None else
                                           f"{r[0]:7.2f} {r[1]:6.2f} {r[2]:7.1f}") for r in row.values()))

def pick(lib, tier):
    atten = TIERS[tier]
    for name, _ in CANDIDATES[lib]:
        rows = sweep[(lib, name)].values()
        if all(r is not None and -r[2] >= atten and abs(r[0] if tier == "economy" else r[1]) <= DROOP_MAX_DB for r in rows):
            return name
    return None

MATCHED = {tier: {lib: pick(lib, tier) for lib in CANDIDATES} for tier in TIERS}
for tier, picks in MATCHED.items():
    print(f"{tier}-matched:", ", ".join(f"{lib} = {p}" for lib, p in picks.items()))
# The benches' constants (bench/compare/bench_compare.cpp, bench/icount/cmp_main.cpp).
assert MATCHED["economy"] == {"libsamplerate": "sinc_medium", "soxr": "16-bit @ 0.750", "r8brain": "70 dB @ 16 %",
                              "SpeexDSP float": "q2", "SpeexDSP fixed": "q3"}, MATCHED["economy"]
assert MATCHED["transparent"] == {"libsamplerate": "sinc_best", "soxr": "20-bit @ 0.800", "r8brain": "120 dB @ 10 %",
                                  "SpeexDSP float": "q9", "SpeexDSP fixed": None}, MATCHED["transparent"]

--- libsamplerate
setting             ↑2 48 → 96: eco    tr  worst    ↓2 96 → 48: eco    tr  worst   3/2 32 → 48: eco    tr  worst   2/3 48 → 32: eco    tr  worst


sinc_fastest         -0.51  -6.27  -111.3     -0.51  -6.27  -107.7     -0.51  -6.27   -98.0     -0.51  -6.27   -98.1


sinc_medium          -0.00  -0.00  -117.2     -0.00  -0.00  -116.9     -0.00  -0.00  -119.3     -0.00  -0.00  -124.3


sinc_best            -0.00  -0.00  -145.2     -0.00  -0.00  -146.0     -0.00  -0.00  -141.5     -0.00  -0.00  -144.0
--- soxr
setting             ↑2 48 → 96: eco    tr  worst    ↓2 96 → 48: eco    tr  worst   3/2 32 → 48: eco    tr  worst   2/3 48 → 32: eco    tr  worst
16-bit @ 0.750        0.00  -1.25  -120.4      0.00  -1.25  -157.8     -0.00  -1.29  -116.3     -0.00  -1.29  -112.5


16-bit @ 0.800       -0.00  -0.08  -122.0     -0.00  -0.08  -150.5     -0.00  -0.09  -114.3     -0.00  -0.09  -118.4
16-bit @ 0.833       -0.00  -0.00  -117.0     -0.00  -0.00  -153.3     -0.00  -0.00  -116.7     -0.00  -0.00  -120.6


...

q9                    0.00  -0.00  -131.6      0.00  -0.00  -131.7     -0.00  -0.00  -135.8     -0.00  -0.00  -130.8


q10                   0.00  -0.00  -134.3      0.00  -0.00  -128.0     -0.00  -0.00  -132.2     -0.00  -0.00  -130.9
--- SpeexDSP fixed
setting             ↑2 48 → 96: eco    tr  worst    ↓2 96 → 48: eco    tr  worst   3/2 32 → 48: eco    tr  worst   2/3 48 → 32: eco    tr  worst
q1                   -0.79  -3.34   -67.7     -1.44  -4.94   -67.3     -0.79  -3.34   -66.3     -1.44  -4.94   -71.1


q2                   -0.00  -0.46   -70.5      0.00  -1.51   -71.3     -0.00  -0.46   -69.4      0.00  -1.51   -73.8
q3                    0.00  -0.06   -99.1      0.00  -0.40   -75.4     -0.00  -0.07   -81.6      0.00  -0.40   -80.2


q4                   -0.00  -0.00   -82.9      0.00  -0.00   -71.5     -0.00  -0.00   -82.9     -0.00   0.00   -74.7
q9                   -0.00  -0.00   -76.9      0.00  -0.01   -76.1     -0.00  -0.00   -75.6     -0.00  -0.00   -78.6


q10                  -0.00  -0.00   -76.4     -0.00  -0.00   -73.8     -0.00   0.00   -75.3     -0.00   0.00   -75.1
economy-matched: libsamplerate = sinc_medium, soxr = 16-bit @ 0.750, r8brain = 70 dB @ 16 %, SpeexDSP float = q2, SpeexDSP fixed = q3
transparent-matched: libsamplerate = sinc_best, soxr = 20-bit @ 0.800, r8brain = 120 dB @ 10 %, SpeexDSP float = q9, SpeexDSP fixed = None


## THD+N and dynamic range at the 24-bit interface, four ratios

997 Hz at −1 dBFS; DR at −60 dBFS, A-weighted; the residual integrated
20 Hz–20 kHz, or to the output Nyquist where that is lower (2/3: 16 kHz).
Every subject converts the same stream at the same ratio; the libraries are
handed the exact rates. The fixed-point rows are the 16-bit interface's
floor, as in the sibling comparisons.

In [4]:
SUBJECTS = [
    ("rational economy",             lambda x, s, d: via_rational(x, s, d, "economy")),
    ("rational balanced",            lambda x, s, d: via_rational(x, s, d, "balanced")),
    ("rational transparent",         lambda x, s, d: via_rational(x, s, d, "transparent")),
    ("libsamplerate FASTEST",        lambda x, s, d: via_lsr(x, s, d, "sinc_fastest")),
    ("libsamplerate MEDIUM (eco)",   lambda x, s, d: via_lsr(x, s, d, "sinc_medium")),
    ("libsamplerate BEST (tr)",      lambda x, s, d: via_lsr(x, s, d, "sinc_best")),
    ("soxr 16-bit @ 0.75 (eco)",     lambda x, s, d: via_soxr(x, s, d, -1, 16.0, 0.75)),
    ("soxr 20-bit @ 0.80 (tr)",      lambda x, s, d: via_soxr(x, s, d, -1, 20.0, 0.80)),
    ("soxr VHQ",                     lambda x, s, d: via_soxr(x, s, d, SOXR_VHQ)),
    ("r8brain 70 dB @ 16 % (eco)",   lambda x, s, d: via_r8brain(x, s, d, 16.0, 70.0)),
    ("r8brain 120 dB @ 10 % (tr)",   lambda x, s, d: via_r8brain(x, s, d, 10.0, 120.0)),
    ("r8brain CDSPResampler24",      lambda x, s, d: via_r8brain(x, s, d, 2.0, 180.15)),
    ("SpeexDSP q2 float (eco)",      lambda x, s, d: via_speex(x, s, d, 2)),
    ("SpeexDSP q9 float (tr)",       lambda x, s, d: via_speex(x, s, d, 9)),
    ("SpeexDSP q10 float",           lambda x, s, d: via_speex(x, s, d, 10)),
    ("rational economy Q15",         lambda x, s, d: via_rational(x, s, d, "economy", "q15")),
    ("rational transparent Q15",     lambda x, s, d: via_rational(x, s, d, "transparent", "q15")),
    ("SpeexDSP q3 fixed-point",      lambda x, s, d: via_speex(x, s, d, 3, True)),
    ("SpeexDSP q10 fixed-point",     lambda x, s, d: via_speex(x, s, d, 10, True)),
]
ANALYZE = 4.0
results = {}   # direction -> name -> (thdn float, thdn 24-bit, dr float, dr 24-bit)
for dname, (src, dst) in DIRECTIONS.items():
    x_full = tone(997.0, ANALYZE + 2.0, src, -1.0)
    x_dr = tone(997.0, ANALYZE + 2.0, src, -60.0)
    results[dname] = {}
    print(f"===== {dname} kHz")
    band = (20.0, min(20000.0, dst / 2 - 1000.0))   # the output Nyquist bounds the band at 2/3
    print(f"{'subject':30s} {'THD+N fl':>9s} {'THD+N 24b':>10s} {'DR fl':>7s} {'DR 24b':>7s}")
    for name, conv in SUBJECTS:
        y = mid(conv(x_full, src, dst), dst, ANALYZE)
        yd = mid(conv(x_dr, src, dst), dst, ANALYZE)
        r = (thdn_db(y, 997.0, dst, band=band), thdn_db(q24(y), 997.0, dst, band=band),
             -thdn_db(yd, 997.0, dst, weighted=True, band=band) + 60.0,
             -thdn_db(q24(yd), 997.0, dst, weighted=True, band=band) + 60.0)
        results[dname][name] = r
        print(f"{name:30s} {r[0]:9.1f} {r[1]:10.1f} {r[2]:7.1f} {r[3]:7.1f}")

# rational's own rows, pinned (24-bit THD+N; the stage contracts in PLAN.md section 6 are the design's,
# these are the shipping engine's through the C ABI).
for dname in DIRECTIONS:
    r = results[dname]
    assert r["rational economy"][1] < -80.0, (dname, r["rational economy"])
    assert r["rational transparent"][1] < -120.0, (dname, r["rational transparent"])
    assert r["rational economy Q15"][1] < -72.0, (dname, r["rational economy Q15"])

===== ↑2 48 → 96 kHz
subject                         THD+N fl  THD+N 24b   DR fl  DR 24b


rational economy                  -152.8     -146.6   213.8   152.1


rational balanced                 -152.8     -146.7   213.7   152.0


rational transparent              -152.9     -146.7   213.7   152.1


libsamplerate FASTEST             -152.2     -146.4   212.9   152.1


...

soxr VHQ                          -152.1     -143.7   211.9   147.6
r8brain 70 dB @ 16 % (eco)        -151.4     -143.6   211.3   147.8
r8brain 120 dB @ 10 % (tr)        -150.5     -143.3   210.7   147.6


r8brain CDSPResampler24           -152.1     -143.7   211.9   147.6
SpeexDSP q2 float (eco)           -139.0     -137.9   199.1   147.6


SpeexDSP q9 float (tr)            -149.8     -143.3   209.4   147.6
SpeexDSP q10 float                -149.2     -143.2   209.9   147.6


rational economy Q15               -94.9      -94.9    98.0    98.0
rational transparent Q15           -95.3      -95.3    97.8    97.8
SpeexDSP q3 fixed-point            -95.2      -95.2    97.9    97.9


SpeexDSP q10 fixed-point           -95.3      -95.3    97.8    97.8


## Latency

In milliseconds, from each engine's own accounting: `rational`'s is its
exact group delay in output frames (`latency_output_frames()`, a rational
number); libsamplerate's is the input frames counted through its streaming
API before the first output; r8brain's is `getInLenBeforeOutPos(0)` in input
frames; SpeexDSP's is `get_input_latency()` in input frames; soxr's is
`soxr_delay()` after priming, in output frames. Each is divided by its own
rate, so the column compares like with like.

In [5]:
# (label, frames function, "in" or "out": the rate the frames are counted at)
LATENCY = [
    ("rational economy",           lambda s, d: rational_latency_output_frames(s, d, "economy"), "out"),
    ("rational transparent",       lambda s, d: rational_latency_output_frames(s, d, "transparent"), "out"),
    ("libsamplerate FASTEST",      lambda s, d: lsr_latency_frames(s, d, "sinc_fastest"), "in"),
    ("libsamplerate MEDIUM (eco)", lambda s, d: lsr_latency_frames(s, d, "sinc_medium"), "in"),
    ("libsamplerate BEST (tr)",    lambda s, d: lsr_latency_frames(s, d, "sinc_best"), "in"),
    ("soxr 16-bit @ 0.75 (eco)",   lambda s, d: _soxr.tap_sr_cmp_soxr_delay_frames(s, d, -1, 16.0, 0.75), "out"),
    ("soxr 20-bit @ 0.80 (tr)",    lambda s, d: _soxr.tap_sr_cmp_soxr_delay_frames(s, d, -1, 20.0, 0.80), "out"),
    ("soxr VHQ",                   lambda s, d: _soxr.tap_sr_cmp_soxr_delay_frames(s, d, SOXR_VHQ, 0.0, 0.0), "out"),
    ("r8brain 70 dB @ 16 % (eco)", lambda s, d: _r8b.tap_sr_cmp_r8b_latency_frames(s, d, 16.0, 70.0, 0), "in"),
    ("r8brain 120 dB @ 10 % (tr)", lambda s, d: _r8b.tap_sr_cmp_r8b_latency_frames(s, d, 10.0, 120.0, 0), "in"),
    ("r8brain CDSPResampler24",    lambda s, d: _r8b.tap_sr_cmp_r8b_latency_frames(s, d, 2.0, 180.15, 0), "in"),
    ("SpeexDSP q2 (eco)",          lambda s, d: _speex.tap_sr_cmp_speex_latency_frames(s, d, 2, 0), "in"),
    ("SpeexDSP q9 (tr)",           lambda s, d: _speex.tap_sr_cmp_speex_latency_frames(s, d, 9, 0), "in"),
    ("SpeexDSP q10",               lambda s, d: _speex.tap_sr_cmp_speex_latency_frames(s, d, 10, 0), "in"),
]
latency = {}   # name -> ratio -> milliseconds
print(f"{'subject':30s} " + "   ".join(f"{d:>12s}" for d in DIRECTIONS))
for name, fn, unit in LATENCY:
    latency[name] = {}
    cells = []
    for dname, (src, dst) in DIRECTIONS.items():
        fr = float(fn(src, dst))
        ms = 1000.0 * fr / (dst if unit == "out" else src)
        latency[name][dname] = ms
        cells.append(f"{ms:9.2f} ms")
    print(f"{name:30s} " + "   ".join(cells))
assert all(latency["rational economy"][d] < 0.5 for d in DIRECTIONS)

subject                          ↑2 48 → 96     ↓2 96 → 48    3/2 32 → 48    2/3 48 → 32
rational economy                    0.22 ms        0.22 ms        0.33 ms        0.33 ms
rational transparent                0.64 ms        0.64 ms        0.77 ms        0.77 ms
libsamplerate FASTEST               0.44 ms        0.42 ms        0.66 ms        0.65 ms
libsamplerate MEDIUM (eco)          1.00 ms        0.97 ms        1.50 ms        1.48 ms
libsamplerate BEST (tr)             3.02 ms        3.00 ms        4.53 ms        4.50 ms
soxr 16-bit @ 0.75 (eco)            9.00 ms        4.50 ms        2.00 ms        4.33 ms
soxr 20-bit @ 0.80 (tr)             5.08 ms        7.00 ms        7.38 ms        4.90 ms
soxr VHQ                            6.38 ms       11.08 ms       24.88 ms       16.58 ms
r8brain 70 dB @ 16 % (eco)          2.29 ms        2.30 ms        2.12 ms        2.12 ms
r8brain 120 dB @ 10 % (tr)          4.33 ms        4.34 ms        9.16 ms        9.17 ms
r8brain CDSPResampler

## Reading the results

- **The matched picks are the bridge comparison's, one notch down.** The
  tiers' passband edges are fractions of the lower rate, so the competitors
  need slightly less than at 44.1 ↔ 48 to keep them: r8brain's 70 dB setting
  holds the edge at a 16 % band (12 % there), soxr at a 0.75 passband
  (0.816), SpeexDSP at quality 2 (3). libsamplerate has no knob between
  `FASTEST` and `MEDIUM`, so `MEDIUM` is its economy row at every ratio.
  Every pick is asserted against the benches' constants.
- **SpeexDSP's quality 2 is the one competitor setting that lands on
  `economy`'s 70 dB** (70–76 dB worst product across the four ratios);
  r8brain's economy-matched setting measures 85–92 dB, libsamplerate's
  `MEDIUM` ~117 dB, soxr's 16-bit ~110–158 dB. The cost tables in
  `docs/COMPARISON.md` say what each pays.
- **A 997 Hz probe measures the imaging floor only where an image lands in
  band.** At ↑2, ↓2 and 2/3 every float row sits at the 24-bit ceiling; at
  3/2 the probe's image lands at 17 kHz and the table separates the tiers:
  `economy` −88 dB, `transparent` −140 dB, the competitors' economy rows at
  −82 (SpeexDSP) to −122 dB. The sweep above is the contract measurement at
  every ratio; this table is its 997 Hz slice.
- **The 16-bit rows are format floors**, as in both sibling comparisons:
  `rational` economy Q15 and SpeexDSP's fixed-point build sit at ~98–101 dB
  A-weighted DR everywhere.
- **Latency.** `rational`'s economy stage is 0.22 ms by 2 and 0.33 ms for
  the mixed pair, the shortest in the table; SpeexDSP's quality 2 is 1.5×
  that, libsamplerate `MEDIUM` ~4.5×, r8brain ~7–10×, soxr 6–40×; the
  frontier settings run to 160–225× (`CDSPResampler24`).

The computational side — host throughput, and executed instructions per
output frame on Cortex-M55, Cortex-M33 and Hexagon with each engine's
construction split out — is in `docs/COMPARISON.md`, measured with the same
matched settings.